# 3. Portfolio evaluation and scientific discussion

Load the saved predictions and frozen selection. This notebook does not train models or choose a new competition entry. Report the assignment and competition experiments separately.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/terminal34-datascience-bootcamp/stock-market-ml/blob/main/03_portfolio_evaluation.ipynb)

## Run locally or in Google Colab

In Colab, select a **Python 3 CPU runtime**, then run the setup cell below and authorize its Google Drive mount. It loads the shared code from GitHub and installs the runtime dependencies. Use the same `EXPERIMENT_NAME` and `COLAB_SMOKE_TEST` values in notebooks **1 → 2 → 3**; each can run in a separate session. Your datasets, models, predictions, and frozen selection are saved directly under `My Drive/stock-market-ml/<experiment>/real/` (or `smoke/`), so they survive runtime resets. Save a notebook copy to Drive if you want to keep your notes and cell outputs too.

**Private GitHub repository:** add a `GITHUB_TOKEN` in Colab's Secrets panel (key icon), using a fine-grained token with **Contents: Read-only** access to this repository, and enable notebook access in each notebook. Organization approval may be required. Do not paste the token into a code cell. Setup passes it only to Git subprocesses and does not save it in artifacts or Git remote URLs. If Colab cannot open the private GitHub link, download the `.ipynb` from GitHub and upload it to Colab; the same secret enables the code download.

The first notebook records a Git commit and exact package versions; later notebooks reuse them. If setup asks for a runtime restart after installing packages, restart the session and rerun setup. Run one notebook at a time for a given experiment. A missing-handoff error means you need to finish the preceding notebook with the same settings. On a local machine, setup preserves your environment and artifact path; install `requirements.txt` first.

Changing the experiment folder after seeing 2026 results does not restore holdout independence. To try this Colab port without revisiting the real holdout, set `COLAB_SMOKE_TEST=True` in all three notebooks. Your existing local artifacts are not uploaded automatically.

In [ ]:
# Use the SAME values in all three Colab notebooks.
EXPERIMENT_NAME = 'competition-v1'
COLAB_SMOKE_TEST = False  # True = synthetic data and small models, stored separately.

"""Standard-library-only bootstrap, embedded into each notebook before third-party imports."""
from pathlib import Path
import base64
import importlib.metadata
import importlib.util
import json
import os
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/terminal34-datascience-bootcamp/stock-market-ml.git'
RUNTIME_PACKAGES = {
    'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy',
    'scikit-learn': 'sklearn', 'xgboost': 'xgboost', 'yfinance': 'yfinance',
    'pyarrow': 'pyarrow', 'matplotlib': 'matplotlib', 'seaborn': 'seaborn', 'joblib': 'joblib',
}


def in_colab():
    try:
        return importlib.util.find_spec('google.colab') is not None
    except (ImportError, ValueError):
        return False


def require_handoff(artifacts, notebook_number):
    required = {
        1: [],
        2: ['data/metadata.json'],
        3: ['frozen_experiment.json', 'competition_evaluation.json',
            'predictions/assignment.parquet', 'predictions/competition.parquet'],
    }
    if notebook_number not in required:
        raise ValueError('Notebook number must be 1, 2, or 3')
    missing = [name for name in required[notebook_number] if not (artifacts / name).is_file()]
    if missing:
        raise RuntimeError(
            f'Missing handoff files in {artifacts}: {missing}. '
            f'Run Notebook {notebook_number - 1} first, with the same EXPERIMENT_NAME and smoke setting.'
        )


def github_git_environment(token=None):
    # Headers live only in child-process environments, never URLs, arguments, or .git/config.
    environment = {key: value for key, value in os.environ.items()
                   if not key.startswith('GIT_TRACE') and key != 'GIT_CURL_VERBOSE'}
    environment['GIT_TERMINAL_PROMPT'] = '0'
    if token:
        encoded = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
        environment.update({'GIT_CONFIG_COUNT': '1',
                            'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
                            'GIT_CONFIG_VALUE_0': f'Authorization: Basic {encoded}'})
    return environment


def prepare_colab_checkout(artifacts, project, repository_url=REPOSITORY_URL, token=None):
    """Use one source revision and resolved package set for all three sessions."""
    artifacts, project = Path(artifacts), Path(project)
    artifacts.mkdir(parents=True, exist_ok=True)
    manifest_path = artifacts / 'colab_environment.json'
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else None
    python_version = f'{sys.version_info.major}.{sys.version_info.minor}'
    if manifest and (manifest['repository_url'] != repository_url or manifest['python'] != python_version):
        raise RuntimeError('This experiment belongs to a different repository or Python version. '
                           'Use the recorded runtime version to reuse its models.')
    git_environment = github_git_environment(token)
    if not project.exists():
        try:
            subprocess.check_call(['git', 'clone', '--quiet', repository_url, str(project)], env=git_environment)
        except subprocess.CalledProcessError as error:
            raise RuntimeError('Cannot clone the repository. For this private repo, add a GITHUB_TOKEN '
                               'Colab secret with Contents: Read access, enable notebook access, and rerun setup.') from error
    if not (project / '.git').is_dir():
        raise RuntimeError(f'{project} exists but is not a Git checkout; choose a fresh Colab session.')
    git = ['git', '-C', str(project)]
    origin = subprocess.check_output(git + ['remote', 'get-url', 'origin'], text=True).strip()
    if origin != repository_url:
        raise RuntimeError(f'Refusing to use an unrelated checkout at {project}')
    if subprocess.check_output(git + ['status', '--porcelain'], text=True).strip():
        raise RuntimeError('The Colab checkout has local edits. Save them before using a fresh runtime; '
                           'setup will not overwrite your work.')
    ref = manifest['commit'] if manifest else 'main'
    subprocess.check_call(git + ['fetch', '--quiet', 'origin', ref], env=git_environment)
    commit = subprocess.check_output(git + ['rev-parse', 'FETCH_HEAD'], text=True).strip()
    subprocess.check_call(git + ['checkout', '--quiet', '--detach', commit])
    requirements = project / 'requirements-colab.txt'
    if not requirements.is_file():
        raise RuntimeError('This GitHub revision does not include Colab support. Publish the updated project first.')
    if manifest:
        requirements = artifacts / 'colab-runtime-requirements.txt'
        requirements.write_text(''.join(f'{name}=={version}\n' for name, version in manifest['packages'].items()))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(requirements)])
    resolved = {name: importlib.metadata.version(name) for name in RUNTIME_PACKAGES}
    if manifest and resolved != manifest['packages']:
        raise RuntimeError('Installed packages do not match the saved Colab environment')
    if manifest is None:
        manifest = {'repository_url': repository_url, 'commit': commit,
                    'python': python_version, 'packages': resolved}
        temporary = manifest_path.with_suffix('.json.tmp')
        temporary.write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n')
        temporary.replace(manifest_path)
        (artifacts / 'colab-runtime-requirements.txt').write_text(
            ''.join(f'{name}=={version}\n' for name, version in resolved.items()))
    changed = [name for name, module in RUNTIME_PACKAGES.items()
               if module in sys.modules and getattr(sys.modules[module], '__version__', resolved[name]) != resolved[name]]
    if changed:
        raise RuntimeError(f'Installed versions changed for already-imported packages: {changed}. '
                           'Choose Runtime > Restart session, then run this setup cell again.')
    print('Pinned GitHub revision:', commit)
    return project


def setup_notebook(notebook_number, experiment_name='competition-v1', colab_smoke=False):
    if in_colab():
        # Authorization is performed by Colab's own Drive prompt, in the user's runtime.
        from google.colab import drive, userdata
        if not experiment_name or experiment_name in ['.', '..'] or any(c in experiment_name for c in '/\\'):
            raise ValueError('EXPERIMENT_NAME must be one folder name')
        drive.mount('/content/drive')
        artifacts = Path('/content/drive/MyDrive/stock-market-ml') / experiment_name / ('smoke' if colab_smoke else 'real')
        require_handoff(artifacts, notebook_number)
        try:
            token = userdata.get('GITHUB_TOKEN')
        except userdata.SecretNotFoundError:
            token = None  # Public forks need no token; private-repo errors explain the setup.
        except userdata.NotebookAccessError as error:
            raise RuntimeError('Enable notebook access for GITHUB_TOKEN in the Colab Secrets panel.') from error
        project = prepare_colab_checkout(artifacts, Path('/content/stock-market-ml'), token=token)
        os.environ['STOCK_ML_ARTIFACTS'] = str(artifacts)
        os.environ['STOCK_ML_SMOKE'] = '1' if colab_smoke else '0'
        os.chdir(project)
    else:
        project = Path.cwd()
        if not (project / 'stockml').is_dir():
            raise RuntimeError('For local execution, open the notebook from the repository root.')
        artifacts = Path(os.environ.get('STOCK_ML_ARTIFACTS', 'artifacts')).resolve()
        require_handoff(artifacts, notebook_number)
    sys.path.insert(0, str(project))
    os.environ.setdefault('MPLCONFIGDIR', str(project / '.cache' / 'matplotlib'))
    Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)
    print('Artifact handoff folder:', artifacts)
    return project

PROJECT = setup_notebook(3, EXPERIMENT_NAME, COLAB_SMOKE_TEST)

import pandas as pd
from IPython.display import display, Markdown
from stockml.common import root, smoke, load_json

pd.set_option('display.max_columns', 20)
print('Artifacts:', root())
print('SYNTHETIC SMOKE TEST — NOT FINANCIAL RESULTS' if smoke() else 'REAL-DATA EXPERIMENT')

In [ ]:
from stockml.models import check_frozen
frozen = check_frozen()
display(pd.Series(frozen['selected_entry']))
display(pd.read_csv(root() / 'reports' / 'validation_portfolios.csv'))

## Allocation, timing, costs, and accounting

| Strategy | Rule |
|---|---|
| Equal weight | One-eighth in each stock; rebalance every five sessions |
| Return-ranked ML | One-third in each of the three highest predicted-return stocks |
| Risk-aware ML | Exclude risk probability ≥0.5, rank the rest, and fill up to three one-third slots |
| SPY | Buy at the first execution open and hold |

Missing risk-aware slots remain in cash. Ties use alphabetical ticker order. Both algorithms and both pooled/individual layouts are reported. Only the entry selected by validation net Sharpe (then lower turnover, then strategy name) is the competition entry.

Each strategy starts with $100,000, uses fractional adjusted units, no leverage, no cash interest, and 10 basis points per dollar bought or sold. Signals from a completed close execute at the next session's open. Rebalancing uses drifted holdings and solves for post-cost wealth before allocating, so costs cannot create borrowing.

Track close-to-close portfolio values daily, including the initial allocation cost and first open-to-close return. Annualize with 252 sessions; use a zero risk-free rate for Sharpe. Drawdown includes the initial $100,000 peak. Terminal wealth is marked to market without liquidation. Turnover is total absolute dollars traded divided by pre-trade wealth at each rebalance; it is a two-sided, cumulative measure.

The required five-day return target is close-to-close; executable portfolio returns start at the following open. They are deliberately not treated as identical. Final portfolio valuation may include sessions whose five-day predictive targets are still unknown.

In [ ]:
from stockml.reporting import report_portfolios, evidence_discussion
assignment_performance, assignment_daily, assignment_trades = report_portfolios('assignment')

In [ ]:
evidence_discussion('assignment')

## Independent competition results

The following chart and table report the frozen entry alongside all predeclared comparators. Do not replace the entry with whichever comparator wins on 2026. A partial-year result is labeled year-to-date and includes its final completed session.

In [ ]:
competition_performance, competition_daily, competition_trades = report_portfolios('competition')

In [ ]:
evidence_discussion('competition')

## Interpretation checklist and limitations

Use the generated evidence above and Notebook 2's importance/ablation tables to defend these conclusions:

1. Which algorithm wins on return RMSE and risk ROC-AUC? Are the winners consistent across stocks?
2. Which task shows more reliable improvement over its own baseline? Do not compare RMSE units with AUC units.
3. Does pooling help each algorithm, given that individual models reuse pooled tuning settings?
4. Does the **frozen** entry outperform SPY after costs, and what happens to drawdown and volatility?
5. Do better predictions correspond to better ranked portfolios? Use the joined prediction/investment table.
6. Does the evidence support generalization across both periods? A short holdout and overlapping labels do not establish statistical significance.
7. Which financial features matter on validation? Compare top-feature identities and the feature ablation before asserting that additional features helped.

The model universe was selected retrospectively and is concentrated in large surviving companies. Regime shifts, Yahoo data revisions, simplified fills, fixed costs, and zero cash yield constrain interpretation. A longer preregistered forward test, point-in-time universe, block-bootstrap uncertainty intervals, and richer execution assumptions would strengthen a future experiment. Do not tune these changes on 2026 after seeing its results.

A successful project can conclude that the models did not beat simple baselines.

## AI assistance and independent verification

AI assistance was used to structure the experiment, draft the shared Python implementation, and create these notebooks and test cases. AI-generated code and conclusions require independent review.

The verification suite checks future-price perturbations, five-day targets against hand calculations, split purging, feature-order rejection, saved-model prediction parity, trade timing, and portfolio accounting against hand-computed constant-price examples. In particular, the transaction-cost implementation uses post-cost wealth rather than allocating all pre-cost cash and implicitly borrowing the fees.

Before submission, run `python -m pytest -q`, inspect these calculations yourself, and add a sentence describing a suggestion or implementation **you personally verified or corrected**. Do not claim personal verification solely because automated tests pass.